## Практическая работа

В этой практической работе четыре обязательные задачи.

*Обязательные задачи* нужно сделать для того, чтобы проверить, что вы действительно усвоили материал модуля. Сдайте их на проверку.

Удачи!

Цели практической работы:


1.   Потренироваться в обучении моделей деревьев решений.
2.   Потренироваться в обучении моделей случайного леса.
3.   Научиться оценивать качество моделей с помошью Accuracy и confusion matrix.
4.   Научиться увеличивать качество моделей с помощью тюнинга параметров.




Что оценивается:

*   Все пункты и критерия приёмки задания выполнены.
*   Удаление колонок по результатам feature_importance и определения типов производится с помощью кода, а не перечислением их вручную.




## Обязательные задачи

### Описание датасета:
- `id`: идентификатор записи;
- `is_manufacturer_name`: признак производителя автомобиля;

- `region_*`: регион;
- `x0_*`: тип топлива;
- `manufacturer_*`: производитель;
- `short_model_*`: сокращённая модель автомобиля;
- `title_status_*`: статус;
- `transmission_*`: коробка передач;
- `state_*`: штат;
- `age_category_*`: возрастная категория автомобиля;

- `std_scaled_odometer`: количество пройденных миль (после стандартизации);
- `year_std`: год выпуска (после стандартизации);
- `lat_std`: широта (после стандартизации);
- `long_std`: долгота (после стандартизации);
- `odometer/price_std`: отношение стоимости к пробегу автомобиля (после стандартизации);
- `desc_len_std`: количество символов в тексте объявления о продаже (после стандартизации);
- `model_in_desc_std`: количество наименований модели автомобиля в тексте объявления о продаже (после стандартизации);
- `model_len_std`: длина наименования автомобиля (после стандартизации);
- `model_word_count_std`: количество слов в наименовании автомобиля (после стандартизации);
- `month_std`: номер месяца размещения объявления о продаже автомобиля (после стандартизации);
- `dayofweek_std`: день недели размещения объявления о продаже автомобиля (после стандартизации);
- `diff_years_std`: количество лет между годом производства автомобиля и годом размещения объявления о продаже автомобиля (после стандартизации);

- `price`: стоимость;
- `price_category`: категория цены.

0. *Подготовка базовой модели*

Обучите простую модель классификации с помощью DecisionTreeClassifier на данных из датасета vehicles_dataset_prepared.csv. Для этого сделайте следующие шаги:

1. Обучите модель дерева решений с зафиксированным random_state на тренировочной выборке.
2. Сделайте предикт на тестовой выборке.
3. Замерьте точность на тестовой выборке и выведите матрицу ошибок.
4. Удалите фичи с нулевыми весами по feature_importance из тренировочной и тестовой выборок.
5. Заново обучите модель и измерьте качество.

In [10]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
df = pd.read_csv('/content/vehicles_dataset_prepared.csv')

df_prepared = df.copy()
df_prepared = df_prepared.drop(['price', 'odometer/price_std'], axis=1)

x = df_prepared.drop(['price_category'], axis=1)
y = df_prepared['price_category']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [3]:
# Ваш код здесь

clf = DecisionTreeClassifier(random_state=42)
clf.fit(x_train, y_train)

pred_train = clf.predict(x_train)
pred_test = clf.predict(x_test)

print(accuracy_score(y_train, pred_train))
print(accuracy_score(y_test, pred_test))

1.0
0.6704781704781705


In [4]:
features = pd.Series(clf.feature_importances_, index=x_train.columns)
features.sort_values(ascending=False)

,0
age_category_new,0.175132
model_len_std,0.080319
std_scaled_odometer,0.080166
desc_len_std,0.069465
year_std,0.050689
...,...
short_model_lr3,0.000000
short_model_lr2,0.000000
short_model_long,0.000000
short_model_limousine,0.000000


In [5]:
df_prepared = df_prepared.drop(features[features == 0 ].index, axis=1)

In [6]:
x = df_prepared.drop(['price_category'], axis=1)
y = df_prepared['price_category']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

clf = DecisionTreeClassifier(random_state=42)
clf.fit(x_train, y_train)

pred_train = clf.predict(x_train)
pred_test = clf.predict(x_test)

print(accuracy_score(y_train, pred_train))
print(accuracy_score(y_test, pred_test))

1.0
0.6725571725571725


In [7]:
confusion_matrix(y_test, pred_test)

array([[730,  54, 213],
       [ 42, 696, 215],
       [192, 229, 515]])

1. *Подготовка модели случайного леса*

Обучите простую модель классификации с помощью RandomForestClassifier. Для этого сделайте следующие шаги:
1. На новых урезанных семплах тренировочной и тестовой выборок обучите модель случайного леса с зафиксированным random_state=50.

2. Сделайте предикт и посчитайте точность модели и матрицу ошибок. Сравните с предыдущей моделью дерева решений. Есть ли случаи, когда модель из пункта 1 отрабатывает лучше, чем модель случайного леса?

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(random_state=50)
rf.fit(x_train, y_train)

pred_train = rf.predict(x_train)
pred_test = rf.predict(x_test)

print(f"Accuracy on training set: {accuracy_score(y_train, pred_train)}")
print(f"Accuracy on test set: {accuracy_score(y_test, pred_test)}")

Accuracy on training set: 1.0
Accuracy on test set: 0.7418572418572419


In [9]:
from sklearn.metrics import confusion_matrix

print("Confusion Matrix for Random Forest:\n", confusion_matrix(y_test, pred_test))

Confusion Matrix for Random Forest:
 [[793  37 167]
 [ 16 789 148]
 [165 212 559]]


ModuleNotFoundError: No module named 'catboost'

Модель случайного леса в нашем случае всегда предсказывает лучше чем первая

2. *Тюнинг модели случайного леса*

Увеличьте точность модели на тестовом датасете RandomForestClassifier c помощью тюнинга параметров.

Параметры, которые можно настраивать для увеличения точности:

```
    `bootstrap'
    'max_depth'
    'max_features'
    'min_samples_leaf'
    'min_samples_split'
    'random_state'
    'n_estimators'

```



С описанием каждого из параметров можно ознакомиться в документации:


https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

Задание засчитывается, если значение метрики строго выше 0,76 на тестовом датасете.

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Исправленная сетка параметров
param_grid = {
   'n_estimators': [100, 300, 500],
   'max_features': ['sqrt', 'log2'],
   'min_samples_leaf': [1, 2, 5],
   'bootstrap': [True, False],
   'max_depth': [10, 20, 30, None],
   'min_samples_split': [2, 5, 10]
}

rf = RandomForestClassifier()
grid_serach_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
    )

grid_serach_rf.fit(x_train, y_train)

best_params = grid_serach_rf.best_params_
best_params

Fitting 5 folds for each of 432 candidates, totalling 2160 fits


AttributeError: 'GridSearchCV' object has no attribute 'best_params'

In [14]:
best_params = grid_serach_rf.best_params_
best_params

{'bootstrap': False,
 'max_depth': None,
 'max_features': 'log2',
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 300}

In [15]:
final_rf = RandomForestClassifier(
    bootstrap=False,
    max_depth=None,
    max_features='log2',
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=300,
    random_state=42 # чтобы результат был стабильным
)

final_rf.fit(x_train, y_train)

RandomForestClassifier(bootstrap=False, max_features='log2', n_estimators=300,
                       random_state=42)

In [16]:
pred_test = final_rf.predict(x_test)
accuracy_score(y_test, pred_test)

0.7664587664587664

3. *Анализ влияния фичей на модель*

Во всех задачах до вы работали над подготовленным датасетом, где фичи были заранее извлечены из текстовых переменных, отскейлены и пропущены через One Hot Encoder. Сравним, какой была бы предсказательная способность модели, если бы мы использовали только сырые данные из исходного датасета. Для этого проделайте следующие шаги:

1. Загрузите датасет `vehicles_dataset_old.csv`.
2. Удалите из него переменную `price` и все строковые колонки. Дерево решений и случайный лес не умеют самостоятельно работать со строковыми значениями.
3. Сформируйте x_train и x_test так же, как они были сформированы в предыдущих заданиях.
4. Обучите свою лучшую модель случайного леса на новых данных и замерьте качество. Убедитесь, что оно ухудшилось.
5. Найдите три фичи, которые лучшим образом повлияли на предсказательную способность модели.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

,id,url,region,region_url,price,year,manufacturer,model,fuel,odometer,title_status,transmission,image_url,description,state,lat,long,posting_date,price_category,date
0,7308295377,https://chattanooga.craigslist.org/ctd/d/chatt...,chattanooga,https://chattanooga.craigslist.org,54990,2020,ram,2500 crew cab big horn,diesel,27442,clean,other,https://images.craigslist.org/00N0N_1xMPvfxRAI...,Carvana is the safer way to buy a car During t...,tn,35.060000,-85.250000,2021-04-17T12:30:50-0400,high,2021-04-17 16:30:50+00:00
1,7316380095,https://newjersey.craigslist.org/ctd/d/carlsta...,north jersey,https://newjersey.craigslist.org,16942,2016,ford,explorer 4wd 4dr xlt,other,60023,clean,automatic,https://images.craigslist.org/00x0x_26jl9F0cnL...,***Call Us for more information at: 201-635-14...,nj,40.821805,-74.061962,2021-05-03T15:40:21-0400,medium,2021-05-03 19:40:21+00:00
2,7313733749,https://reno.craigslist.org/ctd/d/atlanta-2017...,reno / tahoe,https://reno.craigslist.org,35590,2017,volkswagen,golf r hatchback,gas,14048,clean,other,https://images.craigslist.org/00y0y_eeZjWeiSfb...,Carvana is the safer way to buy a car During t...,ca,33.779214,-84.411811,2021-04-28T03:52:20-0700,high,2021-04-28 10:52:20+00:00
3,7308210929,https://fayetteville.craigslist.org/ctd/d/rale...,fayetteville,https://fayetteville.craigslist.org,14500,2013,toyota,rav4,gas,117291,clean,automatic,https://images.craigslist.org/00606_iGe5iXidib...,2013 Toyota RAV4 XLE 4dr SUV Offered by: R...,nc,35.715954,-78.655304,2021-04-17T10:08:57-0400,medium,2021-04-17 14:08:57+00:00
4,7303797340,https://knoxville.craigslist.org/ctd/d/knoxvil...,knoxville,https://knoxville.craigslist.org,14590,2012,bmw,1 series 128i coupe 2d,other,80465,clean,other,https://images.craigslist.org/00F0F_5UAXmOzC18...,Carvana is the safer way to buy a car During t...,tn,35.970000,-83.940000,2021-04-08T15:10:56-0400,medium,2021-04-08 19:10:56+00:00


In [ ]:
df = pd.read_csv('/content/vehicles_dataset_old.csv')
df = df.drop(['price', 'url', 'region', 'region_url', 'manufacturer',
             'model', 'fuel', 'title_status', 'transmission', 'image_url',
             'description', 'state', 'posting_date', 'date'], axis=1)
df

In [ ]:
x = df.drop(['price_category'], axis=1)
y = df.price_category

x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42)

rf = RandomForestClassifier(random_state=42)
rf.fit(x_train, y_train)
pred_train = rf.predict(x_train)
pred_test = rf.predict(x_test)


In [ ]:
print(f"Train accuracy: {accuracy_score(y_train, pred_train):.2f}")
print(f"Test accuracy: {accuracy_score(y_test, pred_test):.2f}")

In [ ]:
features = pd.Series(rf.feature_importances_, index=x_train.columns).sort_values(ascending=False)
features

In [18]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00
0:	learn: 1.0422042	total: 68.7ms	remaining: 1m 8s
100:	learn: 0.5852123	total: 678ms	remaining: 6.03s
200:	learn: 0.5198949	total: 1.24s	remaining: 4.94s
300:	learn: 0.4786623	total: 1.83s	remaining: 4.24s
400:	learn: 0.4504728	total: 2.39s	remaining: 3.56s
500:	learn: 0.4254167	total: 2.97s	remaining: 2.96s
600:	learn: 0.4041114	total: 3.55s	remaining: 2.36s
700:	learn: 0.3858098	total: 4.24s	remaining: 1.81s
800:	learn: 0.3689332	total: 4.9s	remaining: 1.22s
900:	learn: 0.3533530	total: 5.56s	remaining: 611ms
999:	learn: 0.3382204	total: 6.22s	remaining: 0us
Accuracy: 0.7664587664587664


In [23]:
from catboost import CatBoostClassifier

# Создаем модель
cb = CatBoostClassifier(
    iterations=1000,          # аналог n_estimators
    learning_rate=0.1,        # скорость обучения
    depth=10,                  # глубина дерева (у CatBoost 6-10 — это норма)
    verbose=100               # выводить прогресс каждые 100 итераций
)

# Обучаем
cb.fit(x_train, y_train)

# Проверяем
print(f"Accuracy: {cb.score(x_test, y_test)}")


0:	learn: 1.0407275	total: 144ms	remaining: 2m 23s
100:	learn: 0.4893828	total: 8.72s	remaining: 1m 17s
200:	learn: 0.4009320	total: 15.4s	remaining: 1m 1s
300:	learn: 0.3461471	total: 21.3s	remaining: 49.4s
400:	learn: 0.3062774	total: 27.8s	remaining: 41.5s
500:	learn: 0.2715437	total: 33.7s	remaining: 33.5s
600:	learn: 0.2418019	total: 40.6s	remaining: 26.9s
700:	learn: 0.2169528	total: 46.8s	remaining: 20s
800:	learn: 0.1962922	total: 53.2s	remaining: 13.2s
900:	learn: 0.1756850	total: 58.9s	remaining: 6.48s
999:	learn: 0.1600205	total: 1m 5s	remaining: 0us
Accuracy: 0.7716562716562717
